# Dr.DocBench — Comparativo Docling vs MinerU

Notebook de experimentação que compara os dois provedores de extração de
estrutura disponíveis na Toolbox, no contexto do benchmark
[Dr.DocBench](https://drdocbench-challenge.abaka-pages.com/):

- **Docling** (via `docling-serve`, porta 5001) — baseline atual
- **MinerU** (via `mineru-serve`, porta 5002) — incorporado recentemente

Ambos são acessados pela **Toolbox** (porta 8002) através da capability
`document.structure.extract` com o parâmetro `provider`. O payload de cada
provider é convertido para a **árvore canônica** Acessilia
(`build_canonical_document`) e depois para markdown do benchmark
(`canonical_to_drbench_md`).

**Amostra:** 10 páginas do split `dev` com GT local (`adf0ea9c…_p32`–`p41`,
livro de arquitetura — texto corrido, sem tabelas/fórmulas).

**Pré-requisitos:**
- Toolbox rodando (`~/dados/sync/dev/acessilia-toolbox/scripts/run-local.sh`, porta 8002)
- `docling-serve` e `mineru-serve` no ar (docker compose)
- Kernel: `.venv` do acessilia

> Documentação completa: [docs/drbench.pt-br.md](../../drbench.pt-br.md)

In [1]:
# Configuração de caminhos — o pacote de métricas vive em scripts/metrics (dentro do repo)
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "acessilia" and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

VAR_DIR = REPO_ROOT / "var" / "drbench"
GT_DIR = VAR_DIR / "dev"
PRED_DOCLING = VAR_DIR / "predictions"
PRED_MINERU = VAR_DIR / "predictions-mineru"
REPORTS = VAR_DIR / "reports"

print("Repo root:", REPO_ROOT)
print("GT local:", GT_DIR, "→", len(list(GT_DIR.glob("*.md"))), "páginas")

Repo root: /Users/akira/dados/sync/dev/acessilia
GT local: /Users/akira/dados/sync/dev/acessilia/var/drbench/dev → 10 páginas


## 1. Verificação do ambiente

Os providers não rodam neste processo — são serviços HTTP acessados via
Toolbox. Verificamos saúde da Toolbox e dos dois providers.

In [2]:
import httpx

TOOLBOX_URL = "http://localhost:8002"

print("Toolbox:", httpx.get(f"{TOOLBOX_URL}/v1/health", timeout=5).json())

caps = httpx.get(f"{TOOLBOX_URL}/v1/capabilities", timeout=5).json()
for c in caps:
    if c["id"] == "document.structure.extract":
        print("Providers de document.structure.extract:", c["providers"])

Toolbox: {'status': 'healthy', 'version': '0.1.0'}
Providers de document.structure.extract: ['docling', 'mineru']


## 2. Geração das predictions

Para cada provider, rodamos o pipeline completo por página:
imagem → capability `document.structure.extract` → árvore canônica →
`*.drbench.md`.

> As predictions já geradas estão em `var/drbench/predictions` (docling) e
> `var/drbench/predictions-mineru` (mineru). A célula seguinte permite
> regenerar do zero (demora ~2–3 min por provider).

In [3]:
# Regenerar predictions (opcional — descomente para rodar)
# from scripts.drbench.run_pipeline import run_page
# from scripts.drbench.page_record import DrBenchPage
#
# ITEM = "adf0ea9c-f744-4f64-9f74-66a9d276f371"
# for provider, out_dir in [("docling", PRED_DOCLING), ("mineru", PRED_MINERU)]:
#     for p in range(32, 42):
#         page = DrBenchPage.from_item_id(f"{ITEM}_p{p}", split="dev")
#         run_page(page, provider=provider, out_dir=out_dir)

# Inventário atual
for label, d in [("docling", PRED_DOCLING), ("mineru", PRED_MINERU)]:
    files = sorted(d.glob("*.drbench.md"))
    print(f"{label}: {len(files)} predictions em {d.name}")

docling: 13 predictions em predictions
mineru: 10 predictions em predictions-mineru


## 3. Avaliação com as métricas internas

As métricas seguem o contrato EvalAI do Dr.DocBench e vivem em
`scripts/metrics/` (dentro deste repo):

- **text_ed** — Levenshtein normalizado (0–1, menor = melhor)
- **reading_order** — edit distance de sequência de blocos (0–100)
- **teds** — similaridade de árvore de tabelas HTML (0–100)
- **cdm** — F1 token-based de fórmulas LaTeX (0–100)
- **overall** — média dos componentes disponíveis × 100

Componentes sem objetos dos dois lados são *non-scorable* (`None`) e ficam
fora do overall — comportamento idêntico ao EvalAI.

In [6]:
import re

from scripts.metrics.cdm import cdm_score
from scripts.metrics.overall import PageScores, overall_score
from scripts.metrics.reading_order import reading_order_score
from scripts.metrics.teds import teds_score
from scripts.metrics.text_ed import text_ed

TABLE_RE = re.compile(r"<table.*?</table>", re.DOTALL | re.IGNORECASE)
FORMULA_RE = re.compile(r"\$\$(.+?)\$\$", re.DOTALL)


def pairwise_mean(preds, gts, metric):
    """Pair por ordem de aparência; None quando um lado não tem objetos."""
    if not preds or not gts:
        return None
    scores = [metric(p, g) for p, g in zip(preds, gts)]
    return 100.0 * sum(scores) / len(scores) if scores else None


def evaluate(pred_dir):
    pages = {}
    for md_path in sorted(pred_dir.glob("*.drbench.md")):
        item_id = md_path.name.removesuffix(".drbench.md")
        gt_matches = list(GT_DIR.rglob(f"*{item_id}*.md"))
        if not gt_matches:
            continue
        gt_md = gt_matches[0].read_text(encoding="utf-8")
        pred_md = md_path.read_text(encoding="utf-8")

        s = PageScores(item_id=item_id)
        s.text_ed = text_ed(pred_md, gt_md)
        s.reading_order = reading_order_score(
            [b for b in pred_md.split("\n\n") if b.strip()],
            [b for b in gt_md.split("\n\n") if b.strip()],
        )
        s.teds = pairwise_mean(
            TABLE_RE.findall(pred_md), TABLE_RE.findall(gt_md), teds_score
        )
        s.cdm = pairwise_mean(
            FORMULA_RE.findall(pred_md), FORMULA_RE.findall(gt_md), cdm_score
        )
        pages[item_id] = s
    return pages


docling_pages = evaluate(PRED_DOCLING)
mineru_pages = evaluate(PRED_MINERU)

docling_report = overall_score(list(docling_pages.values()))
mineru_report = overall_score(list(mineru_pages.values()))

print("docling:", {k: round(v, 2) for k, v in docling_report.items()})
print("mineru: ", {k: round(v, 2) for k, v in mineru_report.items()})

docling: {'text_ed': 80.89, 'reading_order': 81.42, 'overall': 81.15}
mineru:  {'text_ed': 77.58, 'reading_order': 78.11, 'overall': 77.84}


## 4. Comparação agregada e por página

In [9]:
import pandas as pd

# Agregado — providers no índice, métricas nas colunas
agg = pd.DataFrame(
    {"docling": docling_report, "mineru": mineru_report}
).T[["text_ed", "reading_order", "overall"]].round(2)
agg["delta_mineru"] = (agg.loc["mineru"] - agg.loc["docling"]).round(2)
display(agg)

# Por página
def page_overall(page: PageScores) -> float:
    """Overall de uma única página (mesma regra do agregado)."""
    return overall_score([page]).get("overall", 0.0)

rows = []
for iid in sorted(docling_pages, key=lambda x: int(x.rsplit("p", 1)[-1])):
    d, m = docling_pages[iid], mineru_pages.get(iid)
    rows.append({
        "página": iid.rsplit("p", 1)[-1],
        "docling_overall": round(page_overall(d), 2),
        "mineru_overall": round(page_overall(m), 2) if m else None,
    })
per_page = pd.DataFrame(rows)
per_page["delta"] = (per_page["mineru_overall"] - per_page["docling_overall"]).round(2)
per_page["melhor"] = per_page["delta"].map(
    lambda d: "mineru" if d > 0.01 else ("docling" if d < -0.01 else "empate")
)
display(per_page)

print("\nVitórias: mineru =", (per_page["melhor"] == "mineru").sum(),
      "| docling =", (per_page["melhor"] == "docling").sum(),
      "| empates =", (per_page["melhor"] == "empate").sum())

,text_ed,reading_order,overall,delta_mineru
docling,80.89,81.42,81.15,NaN
mineru,77.58,78.11,77.84,NaN


,página,docling_overall,mineru_overall,delta,melhor
0,32,93.04,88.23,-4.81,docling
1,33,89.36,90.97,1.61,mineru
2,34,71.71,55.66,-16.05,docling
3,35,0.00,0.00,0.00,empate
4,36,98.60,97.52,-1.08,docling
5,37,96.49,96.97,0.48,mineru
6,38,91.88,94.30,2.42,mineru
7,39,82.82,82.99,0.17,mineru
8,40,96.25,95.60,-0.65,docling
9,41,91.37,76.18,-15.19,docling



Vitórias: mineru = 4 | docling = 5 | empates = 1


## 5. Visualização

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Agregado — barras agrupadas manualmente (evita quirks de plot.bar em pandas 3)
metrics = ["text_ed", "reading_order", "overall"]
x_agg = range(len(metrics))
w2 = 0.38
axes[0].bar([i - w2/2 for i in x_agg], [agg.loc["docling", m] for m in metrics],
            w2, label="docling", color="#4c72b0")
axes[0].bar([i + w2/2 for i in x_agg], [agg.loc["mineru", m] for m in metrics],
            w2, label="mineru", color="#dd8452")
axes[0].set_xticks(list(x_agg))
axes[0].set_xticklabels(metrics)
axes[0].set_title("Agregado (10 páginas)")
axes[0].set_ylabel("score (0–100)")
axes[0].set_ylim(60, 85)
for cont in axes[0].containers:
    axes[0].bar_label(cont, fmt="%.2f", fontsize=8)
axes[0].legend()

# Por página
x = range(len(per_page))
w = 0.38
axes[1].bar([i - w/2 for i in x], per_page["docling_overall"], w, label="docling", color="#4c72b0")
axes[1].bar([i + w/2 for i in x], per_page["mineru_overall"], w, label="mineru", color="#dd8452")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(per_page["página"])
axes[1].set_title("Overall por página")
axes[1].set_xlabel("página")
axes[1].set_ylabel("score (0–100)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Comparação das árvores canônicas (p32)

O payload de cada provider é normalizado para a árvore canônica Acessilia
(`provider_payload_to_canonical`). Aqui comparamos as duas árvores da página
p32 lado a lado — estrutura de seções, tipos de bloco e conteúdo.

> Árvores pré-geradas em `var/drbench/canonical-p32-{provider}.json`.
> Para regenerar: `ToolboxClient(provider=...)` + `extract_structure` +
> `provider_payload_to_canonical`.

In [ ]:
import json
from collections import Counter

trees = {}
for provider in ("docling", "mineru"):
    path = VAR_DIR / f"canonical-p32-{provider}.json"
    if path.exists():
        trees[provider] = json.loads(path.read_text(encoding="utf-8"))

for provider, tree in trees.items():
    blocks = [b for s in tree["sections"] for b in s.get("blocks", [])]
    print(f"— {provider}: {len(tree['sections'])} sections, {len(blocks)} blocks",
          dict(Counter(b["type"] for b in blocks)))

def walk(sections, depth=0):
    lines = []
    for s in sections:
        lines.append("  " * depth + f"[S] {s.get('title', '')!r}")
        for b in s.get("blocks", []):
            text = (b.get("text") or "")[:70].replace("\n", " ")
            lines.append("  " * depth + f"    ({b['type']}) {text}")
        lines += walk(s.get("children", []), depth + 1)
    return lines

for provider, tree in trees.items():
    print(f"\n{'=' * 25} {provider} {'=' * 25}")
    print("\n".join(walk(tree["sections"])))

## 7. Análise de diferenças por página

Onde o MinerU perde/ganha em relação ao Docling? Inspecionamos as maiores
diferenças (p34 e p41) comparando blocos presentes no GT e ausentes na
prediction.

In [ ]:
def missing_blocks(item_id: str, pred_dir: Path, top: int = 5) -> list[str]:
    """Blocos do GT ausentes (por similaridade) na prediction."""
    gt_path = GT_DIR / f"{item_id}.md"
    pred_path = pred_dir / f"{item_id}.drbench.md"
    gt_blocks = [b.strip() for b in gt_path.read_text().split("\n\n") if b.strip()]
    pred_blocks = [b.strip() for b in pred_path.read_text().split("\n\n") if b.strip()]
    missing = []
    for gb in gt_blocks:
        best = max(
            (difflib.SequenceMatcher(None, gb[:120], pb[:120]).ratio()
             for pb in pred_blocks),
            default=0.0,
        )
        if best < 0.6:
            missing.append(gb[:90])
    return missing[:top]


import difflib

for iid in ("p34", "p41"):
    item_id = f"adf0ea9c-f744-4f64-9f74-66a9d276f371_{iid}"
    print(f"\n=== {iid}: blocos do GT ausentes na prediction mineru ===")
    for b in missing_blocks(item_id, PRED_MINERU):
        print("  ·", b)

## 8. Latência

Tempo de extração por provider (medido durante a geração das predictions —
reflete o tempo de resposta do serviço, incluindo OCR e layout).

In [ ]:
# Medição ao vivo de uma página (requer Toolbox e providers no ar)
import time
from pathlib import Path as _P

from backend.tools.toolbox_client import ToolboxClient

SAMPLE = GT_DIR / "adf0ea9c-f744-4f64-9f74-66a9d276f371_p32_page_32.jpg"


async def _extract(client: ToolboxClient) -> float:
    start = time.perf_counter()
    await client.extract_structure(file_path=_P(SAMPLE))
    return time.perf_counter() - start


from scripts.drbench.run_pipeline import _run_coro_sync

latency = {}
for provider in ("docling", "mineru"):
    client = ToolboxClient(provider=provider)
    try:
        latency[provider] = _run_coro_sync(_extract(client))
    finally:
        import asyncio
        asyncio.run(client.close())

latency_df = pd.DataFrame(
    {"segundos": {k: round(v, 1) for k, v in latency.items()}}
).T
display(latency_df)

## 9. Conclusões

### Qualidade (overall, 10 páginas texto-corrido)

| Provider | text_ed | reading_order | overall |
|---|---|---|---|
| Docling | 80.89 | 81.42 | **81.15** |
| MinerU | 77.58 | 78.11 | 77.84 |

### Principais achados

1. **MinerU é comparável ao Docling** — gap de ~3 pontos no agregado, com
   vitórias em 4 de 10 páginas (p33, p37, p38, p39).
2. **Perdas pontuais graves** em p34 (−16.1) e p41 (−15.2): o MinerU descarta
   elementos que o Docling preserva — títulos curtos, legendas de figura
   ("Figure 1.7a…") e numeração de página.
3. **Árvores canônicas**: na p32, Docling produziu 7 blocos contra 3 do
   MinerU; ambos classificaram tudo como `paragraph` (nenhum heading —
   fragilidade comum nos dois com páginas de ensaio).
4. **teds/cdm non-scorable** nesta amostra: sem tabelas nem fórmulas. Para
   exercitar esses componentes, ampliar a amostra com páginas do dataset
   que os contenham (o MinerU extrai HTML de tabela e LaTeX nativamente —
   potencial vantagem nesses componentes).
5. **p35 = 0 nos dois** — suspeita de problema no GT (investigar), não nos
   providers.

### Recomendações

- Investigar o GT de p35 antes de qualquer decisão.
- Ampliar a amostra (GT completo do split dev) para decisão de default.
- Avaliar custo de manter os dois providers: os pontos fortes são
  complementares (Docling mais preservacionista; MinerU venceu em páginas
  com layout mais denso).

### Próximos passos sugeridos

- [ ] Baixar GT de páginas com tabelas/fórmulas (`download_gt.py`) e rodar teds/cdm
- [ ] Correr o pipeline PDDL completo com `extractor_backend` de cada provider
- [ ] Avaliar custo operacional (GPU/CPU) de cada serviço